In [ ]:
# AR(1) Disturbance Tuning

import json
import numpy as np
import matplotlib.pyplot as plt

# Load the disturbance model from the simulation notebook
_nb = json.load(open("TEP_Setpoint_Simulation.ipynb"))
for _i in (0, 1, 2, 4):
    exec("".join(_nb["cells"][_i]["source"]))

In [ ]:
# Parameters

HOURS = 50.0        # Length of the generated signals (h)
SEED = 0

DIST_STD = [
    0.001,      # Feed 4 A fraction
    0.00015,    # Feed 4 B fraction
    1.0,        # D feed temp (C)
    1.0,        # C feed temp (C)
    0.5,        # Reactor CW inlet temp (C)
    0.2,        # Condenser CW inlet temp (C)
    0.005,      # Reaction 1 factor
    0.005,      # Reaction 2 factor
    0.02,       # Stripper steam UA factor
    0.01,       # Reactor heat transfer factor
    0.01,       # Condenser heat transfer factor
    0.01,       # Reactor-separator flow factor
]
DIST_TAU = [6.0, 6.0, 6.0, 6.0, 3.0, 3.0, 10.0, 10.0, 6.0, 3.0, 3.0, 3.0]

In [ ]:
# Generate signals

dt = 3.0 / 3600.0
n_steps = int(HOURS / dt)
rec = 60
t = np.arange(0, n_steps + 1, rec) * dt

D = np.zeros((len(t), 12))
plant = TEPlant(SEED, DIST_STD, DIST_TAU)
D[0] = plant.dist
for k in range(1, n_steps + 1):
    plant.update_disturbances(dt)
    if k % rec == 0:
        D[k // rec] = plant.dist

print(f"{'Signal':32s} {'mean':>10s} {'std set':>10s} {'std emp':>10s} {'min':>10s} {'max':>10s}")
for i in range(12):
    d = D[:, i]
    print(f"{DIST_NAMES[i]:32s} {DIST_MEAN[i]:10.4g} {DIST_STD[i]:10.4g} {d.std():10.4g} {d.min():10.4g} {d.max():10.4g}")

In [ ]:
# Plot

fig, axes = plt.subplots(4, 3, figsize=(15, 10), sharex=True)
for i, ax in enumerate(axes.ravel()):
    ax.plot(t, D[:, i], lw=0.8)
    ax.axhline(DIST_MEAN[i], color="k", lw=0.8)
    for s in (-2, 2):
        ax.axhline(DIST_MEAN[i] + s * DIST_STD[i], color="k", ls="--", lw=0.6)
    ax.set_title(f"{i + 1}. {DIST_NAMES[i]} (std {DIST_STD[i]:g}, tau {DIST_TAU[i]:g} h)", fontsize=9)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("Time (h)", fontsize=8)
fig.suptitle("AR(1) disturbance signals (dashed: mean +/- 2 std)")
fig.tight_layout()
plt.show()